# Multi-Agent Systems

**What you will build:** a **supervisor** that routes each request to the right specialist agent, and each specialist handles its own domain. Maps to chapter 10 of the n8n course — including its counter-lesson about when *not* to use multiple agents.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/26_multi_agent_supervisor.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## Specialists as tools

The multi-agent pattern least likely to break across versions is also the simplest: build each specialist with `create_agent`, then expose each one to a **supervisor** as a tool. The supervisor is itself just an agent (2.0) whose tools happen to be other agents. Delegation is tool calling — the same protocol from 0.0, one level up.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

# --- Specialist 1: math ---
@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b
@tool
def percent_of(part: float, whole: float) -> float:
    """Return part as a percentage of whole."""
    return 100 * part / whole

math_agent = create_agent(model=llm, tools=[add, percent_of],
                          system_prompt="You are a math specialist. Use your tools to compute exact answers.")

# --- Specialist 2: research (fake knowledge) ---
@tool
def lookup_capital(country: str) -> str:
    """Return the capital city of a country."""
    caps = {"japan": "Tokyo", "france": "Paris", "spain": "Madrid"}
    return caps.get(country.lower(), "unknown")

research_agent = create_agent(model=llm, tools=[lookup_capital],
                              system_prompt="You are a research specialist. Look up facts with your tools.")

Now wrap each specialist as a delegation tool, and give both to the supervisor:

In [ ]:
@tool
def math_expert(query: str) -> str:
    """Delegate math and calculation questions to the math specialist."""
    out = math_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return out["messages"][-1].content

@tool
def research_expert(query: str) -> str:
    """Delegate factual lookup questions to the research specialist."""
    out = research_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return out["messages"][-1].content

supervisor = create_agent(
    model=llm, tools=[math_expert, research_expert],
    system_prompt="Route each part of the request to the right expert, then combine their answers.",
)

result = supervisor.invoke({"messages": [{"role": "user",
          "content": "What is 30 as a percentage of 240, and what's the capital of Japan?"}]})
print(result["messages"][-1].content)

The supervisor split the request, sent the math to `math_agent` and the fact to `research_agent`, and merged the results. Each specialist stays simple and focused; the supervisor only coordinates.

## The limitation nobody mentions: context loss

Agents-as-tools is the simplest multi-agent pattern, and it has one real weakness worth naming out loud. Look at what actually crosses each boundary:

- **Going down:** the supervisor calls `math_expert(query="...")`. The specialist sees only *that one string* — not the conversation, not the user's earlier turns, not what the other specialist found. The supervisor has to re-describe the task from scratch, and anything it forgets to mention is gone.
- **Coming back:** the specialist returns a *summary* string. Its reasoning, its intermediate tool results, its uncertainty — all squeezed back through the same keyhole.

Information is lossy at every hop. For independent sub-tasks (a calculation, a lookup) that's fine — the keyhole is wide enough. But when a specialist needs the *full picture* to do its job, nesting-and-summarizing quietly drops the context it needed. The alternative is a **handoff**: instead of calling a specialist and waiting for a summary, one agent *transfers control* to another and hands over the shared state — no keyhole.

In [ ]:
# A minimal handoff, as a graph. `triage` inspects the request and TRANSFERS control
# to a specialist by returning a Command — it doesn't call-and-wait, and its turn ends there.
# The specialist runs on the SAME messages (full context), not a re-typed query string.
from typing import Literal
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command

def triage(state: MessagesState) -> Command[Literal["billing", "tech"]]:
    text = state["messages"][-1].content.lower()
    dest = "billing" if any(w in text for w in ("refund", "invoice", "charged", "payment", "plan")) else "tech"
    return Command(goto=dest)          # hand off: no summary in between, full state travels along

def billing(state: MessagesState) -> dict:
    r = llm.invoke([{"role": "system", "content": "You are the billing specialist. Be brief."},
                    *state["messages"]])
    return {"messages": r}

def tech(state: MessagesState) -> dict:
    r = llm.invoke([{"role": "system", "content": "You are the tech-support specialist. Be brief."},
                    *state["messages"]])
    return {"messages": r}

hb = StateGraph(MessagesState)
hb.add_node("triage", triage)
hb.add_node("billing", billing)
hb.add_node("tech", tech)
hb.add_edge(START, "triage")
hb.add_edge("billing", END)
hb.add_edge("tech", END)
handoff = hb.compile()

r = handoff.invoke({"messages": [{"role": "user", "content": "I was charged twice for my Pro plan this month."}]})
print(r["messages"][-1].content)

Two ways to combine agents, two trade-offs:

| | Agents-as-tools (the supervisor above) | Handoff with `Command` |
|---|---|---|
| Control | Supervisor stays in charge, delegates and resumes | Control is *transferred*; the sender's turn ends |
| Context | Specialist sees a re-typed `query`; returns a summary | Specialist sees the full shared state |
| Best for | Independent sub-tasks the supervisor stitches together | Routing/escalation where the next agent needs everything |

Neither is "the multi-agent framework" — both are just the graph primitives from 2.1–2.2 (`Command` is the same return-a-dict-to-update-state, plus a `goto`). Reach for a handoff when the keyhole hurts; keep agents-as-tools when it doesn't.

```{important}
**Multi-agent is not automatically better.** Every delegation is extra model calls, latency, and cost, and more places to go wrong. Reach for it only when domains are genuinely separate or one prompt has become an unmanageable list of rules. Very often a single well-built agent with good tools (Block 1) beats a committee. This is the same warning the n8n course gives in chapter 10.
```

::::{dropdown} 🔧 Under the hood: supervisor patterns
:color: info

There's also a dedicated `langgraph-supervisor` package (`create_supervisor(...)`) and a peer-to-peer `langgraph-swarm`. They add message-passing conventions and hand-off tooling on top of what you just built. The tools-as-delegation pattern here is the most transparent one, which is why we teach it first. For deep multi-agent systems, explore those packages once this pattern is second nature.
::::

```{note}
Two protocols, two levels. **MCP** (18) standardizes agent ↔ *tool*. **A2A** (Agent-to-Agent) standardizes agent ↔ *agent* — a wire format so agents from different teams or frameworks can delegate to each other over a network, instead of the in-process tool-delegation you built here. You rarely need it until agents live in separate services; then it's the MCP of multi-agent.
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Build the single-agent baseline.** Give *one* `create_agent` all three tools (`add`, `percent_of`, `lookup_capital`) and send it the same two-part question the supervisor got. **Done when:** it returns a correct answer with no supervisor and no delegation — proof that for this task the committee was optional.
2. **Price the two designs.** Time both with `time.perf_counter()` and print each answer. **Done when:** you can state the latency multiplier (the supervisor is slower) and explain *why* in one sentence — every delegation is an extra round-trip, and each specialist re-pays for its own system prompt.
3. **Find where the supervisor earns its keep.** Change the task so a single agent's prompt would balloon (e.g. add 3 more domains, each with its own rules and 3 tools). **Done when:** you can finish "I'd split this into agents because…" pointing at prompt size / tool confusion, not at raw speed.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 & 2 —** one agent with every tool, timed against the supervisor:

```python
import time

flat_agent = create_agent(
    model=llm, tools=[add, percent_of, lookup_capital],
    system_prompt="Answer the question, using your tools for any math or lookups.")

q = "What is 30 as a percentage of 240, and what's the capital of Japan?"

t0 = time.perf_counter()
a = flat_agent.invoke({"messages": [{"role": "user", "content": q}]})
t_flat = time.perf_counter() - t0

t0 = time.perf_counter()
b = supervisor.invoke({"messages": [{"role": "user", "content": q}]})
t_sup = time.perf_counter() - t0

print(f"single agent : {t_flat:5.1f}s  {a['messages'][-1].content[:60]}")
print(f"supervisor   : {t_sup:5.1f}s  {b['messages'][-1].content[:60]}")
print(f"latency cost : {t_sup / t_flat:.1f}x")
```

Both answer correctly; the supervisor typically runs ~1.5–3× slower because delegating a sub-task is a whole extra agent round-trip (supervisor decides → specialist runs → supervisor reads the summary → supervisor answers), and each specialist re-sends its own system prompt. For two independent lookups, that overhead buys you nothing.

**3 —** The break-even isn't speed, it's *manageability*. When one agent would need a 40-rule system prompt and 12 tools it keeps confusing, splitting into focused specialists makes each one reliable — you trade latency for correctness and maintainability. That's the same "when NOT to use multiple agents" line from 0.4 and n8n ch.10, now with a number attached.
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| Supervisor answers itself instead of delegating | Delegation tool descriptions too vague | Make each `@tool` docstring say *exactly* which questions it owns ("math and calculations", "factual lookups") |
| `Command(goto=...)` raises "unknown node" | Destination not declared to the graph | Annotate the node's return as `Command[Literal["a", "b"]]` so LangGraph knows the possible targets |
| Handoff specialist ignores earlier turns | You passed a fresh query instead of `*state["messages"]` | In a handoff, forward the full `messages` — that's the whole point of not summarizing |
| Specialist's tool result never reaches the user | Supervisor summarized it away | Expected: agents-as-tools is lossy. If the detail matters, hand off instead of nesting |
::::

**Block 2 core complete.** You can now build stateful graphs: routing, the agent loop, persistence, human-in-the-loop, cyclic reflection, and multi-agent supervision — the tools for genuinely complex, controllable systems.

**What's next:** **27** is an optional capstone — RAG with a self-grading loop (**self-corrective RAG**) — and **28** adds **long-term memory** (remembering a user across separate conversations). Then Block 3 takes an agent to production.